# Student 2 · arXiv embeddings and retrieval

Run the sample to compare two encoders, build three FAISS indexes, and inspect a live result. Full mode encodes Student 1’s clean Parquet on a Kaggle GPU and writes transferable checkpoints for CPU indexing on the demo laptop.

On Kaggle, import this notebook and attach the project ZIP or its extracted folder as a dataset. Enable Internet for the first model download. For the full run, also attach the cleaned Parquet and enable a GPU accelerator.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import zipfile

MODE = "sample"                 # "sample" or "full"
PROJECT_PATH = None             # Optional path to the folder that contains src/embed.py
FULL_PARQUET = None             # Full mode: file or directory of Spark part files
SELECTION_FILE = None           # Full mode: selection.json from the team's development questions
RESUME_FROM = None              # Optional previously saved full work folder containing catalog.sqlite
GPU_BATCH_SIZE = 128            # Reduce this if the GPU runs out of memory
SHARD_ROWS = 200_000            # Keep this fixed when resuming

## Locate the project and install dependencies

The project can run from an extracted local folder or a read-only Kaggle input. Model files download once per encoder.

In [ ]:
cwd = Path.cwd().resolve()
kaggle_input = Path("/kaggle/input")
on_kaggle = kaggle_input.is_dir()
candidates = [Path(PROJECT_PATH)] if PROJECT_PATH else [cwd, cwd.parent, cwd / "rag-arxiv"]
if on_kaggle and not PROJECT_PATH and not any((p / "src" / "embed.py").is_file() for p in candidates):
    candidates.extend(p.parent.parent for p in kaggle_input.rglob("embed.py")
                      if p.parent.name == "src" and (p.parent / "run_sample.py").is_file())
project = next((p for p in candidates if (p / "src" / "run_sample.py").is_file()), None)
if project is None and on_kaggle and not PROJECT_PATH:
    archive = next(kaggle_input.rglob("Student2_arXiv_Retrieval.zip"), None)
    if archive:
        destination = Path("/kaggle/working/student2_code")
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(destination)
        project = destination / "rag-arxiv"
if project is None or not (project / "requirements.txt").is_file():
    raise FileNotFoundError("Attach the project or set PROJECT_PATH to its extracted rag-arxiv folder.")
project = project.resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(project / "requirements.txt")], check=True)
print("Project folder:", project)

## Choose the input and the output directory

The sample uses 18 development questions linked to the attached papers. Full mode accepts the model decision recorded by `src.compare_models`. Leaving `SELECTION_FILE` empty uses the included provisional sample decision.

In [ ]:
import torch

if MODE not in {"sample", "full"}:
    raise ValueError("MODE must be 'sample' or 'full'.")
base = Path("/kaggle/working") if on_kaggle else project / "artifacts"
work = base / ("arxiv_student2_" + MODE)
device = "cuda" if torch.cuda.is_available() else "cpu"

def run_module(name, *arguments):
    command = [sys.executable, "-m", "src." + name, *map(str, arguments)]
    subprocess.run(command, cwd=project, check=True)

if MODE == "full":
    if FULL_PARQUET is None or not Path(FULL_PARQUET).exists():
        raise FileNotFoundError("Set FULL_PARQUET to Student 1's cleaned Parquet file or directory.")
    if device != "cuda":
        raise RuntimeError("Enable a Kaggle GPU accelerator for the full-corpus run.")
    decision = Path(SELECTION_FILE) if SELECTION_FILE else project / "results" / "sample" / "selection.json"
    selected = json.loads(decision.read_text())["model"]
    print("Selected model:", selected, "from", decision)
    if RESUME_FROM and not work.exists():
        previous = Path(RESUME_FROM)
        if not (previous / "catalog.sqlite").is_file():
            raise ValueError("RESUME_FROM must point to the work folder containing catalog.sqlite.")
        work.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=work.parent) as temp:
            restored = Path(temp) / "restored"
            shutil.copytree(previous, restored)
            restored.rename(work)
print("Encoder device:", device, "| Output:", work)

## Run the pipeline

Sample mode rebuilds the local indexes and measures each one on CPU. Full mode encodes only the selected model. Each completed `.npy` shard has a row range and checksum in `manifest.json`. A retry reuses verified shards.

In [ ]:
if MODE == "sample":
    run_module("run_sample", "--work", work, "--device", device)
else:
    run_module("embed", "--data", FULL_PARQUET, "--work", work, "--model", selected,
               "--device", "cuda", "--batch-size", GPU_BATCH_SIZE, "--chunk-size", SHARD_ROWS)

## Inspect the results

In sample mode, the first table compares the encoders against the source paper for each question. The second compares approximate neighbors with exact Flat neighbors. It also reports index RAM and per-query search time.

In [ ]:
import csv
from IPython.display import Markdown, display

if MODE == "sample":
    for filename in ("model_comparison.csv", "index_benchmark.csv"):
        with (work / "reports" / filename).open(newline="", encoding="utf-8") as file:
            rows = list(csv.DictReader(file))
        keys = (["model", "recall_at_1", "recall_at_5", "recall_at_10", "mrr"]
                if filename.startswith("model") else
                ["index", "build_seconds", "ram_mib", "file_mib", "mean_search_ms", "recall_at_10_vs_flat"])
        def format_cell(value):
            try:
                return f"{float(value):.4f}"
            except ValueError:
                return value
        lines = ["| " + " | ".join(keys) + " |", "|" + "|".join(["---"] * len(keys)) + "|"]
        for row in rows:
            lines.append("| " + " | ".join(format_cell(row[key]) for key in keys) + " |")
        display(Markdown("\n".join(lines)))
else:
    manifest = json.loads((work / selected / "embeddings" / "manifest.json").read_text())
    completed = sum(part["stop"] - part["start"] for part in manifest["parts"].values())
    print(f"Completed {completed:,} of {manifest['config']['rows']:,} documents in {len(manifest['parts'])} shards.")

## Send the full run to the demo laptop

Save a Kaggle notebook version with its outputs, or download the completed work folder while the session is active. To resume in a later session, attach that saved folder and set `RESUME_FROM`. Files in a temporary session must be exported or saved before the session disappears.

Copy the entire full work folder to `artifacts/full/` on the laptop. Keep `catalog.sqlite`, the selected model’s `embeddings/manifest.json` and every `.npy` shard together. Put Student 4’s English benchmark questions in `eval/questions.json`. From the project root run the commands below.

```bash
python -m src.build_index --work artifacts/full
python -m src.benchmark --work artifacts/full --questions eval/questions.json
python -m src.retriever --work artifacts/full "How can reader behavior help recommend related papers?" -k 3
```

The benchmark writes the CSV and the Stage 4 index table to `artifacts/full/reports/`. The `README.md` explains the Student 3 integration and lists the files to share with Student 4.